# Урок 7 · Практична робота № 1 · Свій фотофільтр
**Як бачить машина · 9-А · робочий зошит**

**Спершу:** *Файл → Зберегти копію на Диску*. Без цього ваші зміни зникнуть після уроку.

**Як працювати**
* Кроки йдуть так само, як на слайдах: слайд «Зошит · крок 2» = розділ «Крок 2» тут.
* ▶ ліворуч від клітинки — запустити її. Клітинки запускаємо **згори вниз**.
* `___` — пропуск: його заповнюєте ви.
* 💡 **Підказка** — клітинка з прихованим кодом. Відкривайте, лише якщо застрягли.
* **Своє фото:** панель 📁 ліворуч → завантажити → назвати **foto.jpg**. Немає — візьметься тестове.

In [ ]:
#@title Службове: тестове фото й показ (можна не відкривати) { display-mode: "form" }
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def test_photo():
    """Тестове кольорове фото 640×480: небо, сонце, пагорб, будиночок і піщана доріжка."""
    h, w = 480, 640
    y, x = np.mgrid[0:h, 0:w]
    t = y / 300
    img = np.zeros((h, w, 3))
    img[..., 0] = 110 + 60 * t                       # небо: синє вгорі, світліше донизу
    img[..., 1] = 160 + 45 * t
    img[..., 2] = 220 + 20 * t
    img[(x - 500)**2 + (y - 90)**2 < 50**2] = (250, 220, 120)   # сонце
    hill = y > 300 + 30 * np.sin(x / 80)
    img[hill] = (70, 140, 60)                        # пагорб
    img[(x > 90) & (x < 250) & (y > 230) & (y < 340)] = (200, 60, 50)    # стіна будиночка
    roof = (y > 170) & (y <= 230) & (np.abs(x - 170) < (y - 170) * 1.6)
    img[roof] = (110, 60, 40)                        # дах
    path = (y > 360) & (np.abs(x - 330) < (y - 360) * 0.9 + 25)
    img[path] = (230, 200, 150)                      # піщана доріжка
    return img.astype(np.uint8)

def load(name='foto.jpg'):
    """Своє фото, якщо завантажили foto.jpg; інакше — тестове. Велике зменшуємо до 640 пікс."""
    if os.path.exists(name):
        im = Image.open(name).convert('RGB')
        im.thumbnail((640, 640))
        return np.array(im)
    return test_photo()

def show(*imgs, titles=None):
    """Показати кілька кольорових фото поруч."""
    fig, ax = plt.subplots(1, len(imgs), figsize=(4.5 * len(imgs), 3.4))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(im)
        ax[i].axis('off')
        if titles: ax[i].set_title(titles[i])
    plt.show()


## Крок 0 · Беремо фото
Кольорове фото — це **три** таблиці чисел: червона, зелена й синя (урок 4).

In [ ]:
img = load()                       # своє foto.jpg або тестове фото
show(img)
print(img.shape)                   # (висота, ширина, 3 канали)
print('піксель доріжки:', img[440, 330])

## Крок 1 · Сепія
Кожен новий канал — це суміш старих R, G і B зі своїми коефіцієнтами. Спершу перевіряємо на одному пікселі, потім — на всьому фото.

In [ ]:
def sepia(img):
    f = img.astype(float)                       # дроби: 0,769 · 200 — не ціле число
    R, G, B = f[..., 0], f[..., 1], f[..., 2]   # три таблиці: червона, зелена, синя
    r = 0.393 * R + 0.769 * G + 0.189 * B
    g = 0.349 * R + 0.686 * G + 0.168 * B
    b = 0.272 * R + 0.534 * G + 0.131 * B
    new = np.stack([r, g, b], axis=-1)          # знову складаємо три канали в одне фото
    return np.clip(new, 0, 255).astype(np.uint8)   # що більше за 255 — обрізаємо

px = np.array([[[230, 200, 150]]], dtype=np.uint8)   # один піксель — як в анімації
print(sepia(px))

In [ ]:
sep = sepia(img)
show(img, sep, titles=['було', 'сепія'])

✍️ **Ваша відповідь:** яке число вийшло для червоного каналу й чому не 272?

*(пишіть тут)*

## Крок 2 · Постеризація
Ріжемо шкалу 0…255 на сходинки по `step` і кожне число опускаємо на початок його сходинки. Заповніть `___`.

In [ ]:
def poster(img, step=32):
    return img // ___ * ___                     # // — ділення без остачі

print(poster(np.array([200, 210, 223, 224, 31, 32])))
pos = poster(img)
show(img, pos, titles=['було', 'постеризація'])

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# Ділимо без остачі на крок і множимо назад на крок:
# return img // step * step
# 200 // 32 = 6,  6 * 32 = 192

✍️ **Ваша відповідь:** чому 200, 210 і 223 стали однаковими?

*(пишіть тут)*

## Крок 3 · Свій фільтр — практична робота № 1
Оберіть фільтр (або поєднайте два), застосуйте до **свого** фото і збережіть пару «до / після».

In [ ]:
def contrast(img):
    lo, hi = int(img.min()), int(img.max())     # найтемніше й найсвітліше число на фото
    f = (img.astype(float) - lo) * 255 / (hi - lo)   # lo → 0, hi → 255 (урок 5)
    return f.astype(np.uint8)

out = ___                                       # ваш фільтр: sepia(img), poster(img), contrast(img)…
show(img, out, titles=['до', 'після'])

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# Будь-який із трьох:  out = sepia(img)   out = poster(img, 64)   out = contrast(img)
# Або поєднайте два:   out = poster(sepia(img))
# Свій варіант: поміняйте коефіцієнти в sepia() чи крок у poster().

In [ ]:
pair = np.hstack([img, out])                    # «до» і «після» в одну картинку
Image.fromarray(pair).save('do_pislia.jpg')
print('Збережено do_pislia.jpg — панель 📁 ліворуч → ⋮ → Завантажити')

## Крок 4 · Опис голосовим введенням
1. Створіть Google Документ **«ПР 1 — Прізвище Імʼя»**.
2. *Інструменти → Голосовий ввід* (або `Ctrl + Shift + S`, працює в Chrome), мова — **українська**.
3. Натисніть мікрофон і **надиктуйте три речення**: що робить фільтр · яка формула · що вийшло на фото.
4. Виправте помилки розпізнавання **руками** — їх буде кілька, це нормально.
5. Вставте картинку `do_pislia.jpg` (*Вставка → Зображення → Завантажити з компʼютера*).

✍️ **Які слова машина розпізнала неправильно?** Запишіть 2–3 приклади: що казали → що вийшло.

*(пишіть тут)*

## ДЗ · Доробити практичну
* Доробіть фільтр і документ: три речення, формула, пара «до / після».
* Покладіть документ у спільну теку класу (посилання — у чаті Teams).